# MALTO Text Authorship Detection — EDA Notebook

**Purpose:** Exploratory Data Analysis only. No production logic here.

Covers:
- Class distribution
- Text length distribution
- Token count distribution
- Duplicate and empty text detection
- Train vs. test text length comparison
- Word frequency plots per class
- Baseline TF-IDF + LogisticRegression quick check

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Ready.')

## 1. Load Data

In [ ]:
from src.data import load_train, load_test
from src.constants import LABEL_MAP, LABEL_NAMES

train = load_train('../data/train.csv')
test = load_test('../data/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
train.head(3)

## 2. Class Distribution

In [ ]:
label_counts = train['label'].value_counts().sort_index()
label_counts.index = [LABEL_MAP[i] for i in label_counts.index]

fig, ax = plt.subplots()
label_counts.plot(kind='bar', ax=ax, color=sns.color_palette('tab10', 6))
ax.set_title('Class Distribution (Train)')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
for i, v in enumerate(label_counts):
    ax.text(i, v + 5, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(label_counts.to_string())

## 3. Text Length Distribution

In [ ]:
train['char_len'] = train['text'].str.len()
train['word_count'] = train['text'].str.split().str.len()
test['char_len'] = test['text'].str.len()
test['word_count'] = test['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['char_len'], bins=60, alpha=0.7, label='Train', color='steelblue')
axes[0].hist(test['char_len'], bins=60, alpha=0.7, label='Test', color='orange')
axes[0].set_title('Character Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].legend()

axes[1].hist(train['word_count'], bins=60, alpha=0.7, label='Train', color='steelblue')
axes[1].hist(test['word_count'], bins=60, alpha=0.7, label='Test', color='orange')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Words')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Train char length stats:')
print(train['char_len'].describe().round(1))
print('\nTest char length stats:')
print(test['char_len'].describe().round(1))

## 4. Text Length by Class

In [ ]:
train['class_name'] = train['label'].map(LABEL_MAP)

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=train, x='class_name', y='word_count',
            order=LABEL_NAMES, palette='tab10', ax=ax)
ax.set_title('Word Count per Class')
ax.set_xlabel('Class')
ax.set_ylabel('Word Count')
plt.tight_layout()
plt.show()

## 5. Duplicate and Empty Detection

In [ ]:
n_duplicates = train.duplicated(subset=['text']).sum()
n_empty = (train['text'].str.strip() == '').sum()
n_null = train['text'].isna().sum()

print(f'Duplicate texts  : {n_duplicates}')
print(f'Empty strings    : {n_empty}')
print(f'Null values      : {n_null}')

if n_duplicates > 0:
    print('\nSample duplicates:')
    dup_mask = train.duplicated(subset=['text'], keep=False)
    print(train[dup_mask][['text', 'label']].head(4).to_string())

## 6. Top Word Frequencies per Class

In [ ]:
from collections import Counter

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (label_id, label_name) in enumerate(LABEL_MAP.items()):
    class_texts = train[train['label'] == label_id]['text']
    words = ' '.join(class_texts).lower().split()
    # Remove short/common words for better visualisation
    words = [w for w in words if len(w) > 3 and w.isalpha()]
    top_words = Counter(words).most_common(15)
    
    if top_words:
        terms, freqs = zip(*top_words)
        axes[i].barh(list(terms)[::-1], list(freqs)[::-1],
                     color=sns.color_palette('tab10')[i])
        axes[i].set_title(f'{label_name} — Top 15 Words')
        axes[i].set_xlabel('Frequency')

plt.suptitle('Top Words per Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Quick Baseline: TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42, n_jobs=-1)),
])

X_train = train['text'].tolist()
y_train = train['label'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(baseline_pipeline, X_train, y_train,
                         cv=skf, scoring='f1_macro', n_jobs=-1)

print(f'Baseline TF-IDF + LR — Macro F1:')
print(f'  Fold scores : {[round(s, 4) for s in scores]}')
print(f'  Mean ± std  : {scores.mean():.4f} ± {scores.std():.4f}')

## 8. Notes

- For production training, use `python main_train.py --config configs/config.yaml`
- This notebook is for exploration only
- The full pipeline uses word + char ngram TF-IDF combined with FeatureUnion